In [ ]:
# Section 1: Import Required Libraries and Setup

import sys
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path

# Add brent-detector to path
sys.path.insert(0, '/home/awow/personal-work/brent-detector')

from brent.scanner import get_python_files
from brent.parser import extract_imports
from brent.graph_builder import build_dependency_graph
from brent.metrics import calculate_metrics
from brent.brent_ranker import rank_brents

print("✅ All libraries imported successfully")
print(f"Python version: {sys.version}")
print(f"NetworkX version: {nx.__version__}")

# Brent Detector - Comprehensive Testing & Validation

This notebook tests the Brent Detector FYP tool to identify critical software modules in Python codebases using dependency graph analysis.

## Overview
- **Status**: Testing dependency graph construction
- **Issue Found**: 0 dependencies detected in Flask (investigating module matching)
- **Goal**: Validate fixes and ensure proper dependency extraction

## Section 2: Load and Analyze Flask Project

Test the Brent Detector on the Flask framework to identify critical modules.

In [ ]:
# Analyze Flask Project
project_path = "/home/awow/personal-work/brent-detector/data/flask"

print("=" * 60)
print("FLASK PROJECT ANALYSIS")
print("=" * 60)

# Step 1: Scan files
print("\n[1/4] Scanning Python files...")
files = get_python_files(project_path)
print(f"✅ Found {len(files)} Python files")

# Show sample files
print("\nSample files (first 10):")
for f in files[:10]:
    print(f"  - {os.path.relpath(f, project_path)}")

# Step 2: Sample imports
print("\n[2/4] Analyzing imports from sample files...")
sample_imports = {}
for f in files[:5]:
    imports = extract_imports(f, project_path)
    rel_path = os.path.relpath(f, project_path)
    if imports:
        sample_imports[rel_path] = imports[:3]  # First 3 imports
        print(f"\n{rel_path}")
        print(f"  Imports: {imports[:3]}")
    else:
        print(f"\n{rel_path}")
        print(f"  Imports: (none)")

# Step 3: Build graph
print("\n[3/4] Building dependency graph...")
graph = build_dependency_graph(files, extract_imports, project_path)
print(f"✅ Graph created:")
print(f"   Nodes (modules): {graph.number_of_nodes()}")
print(f"   Edges (dependencies): {graph.number_of_edges()}")

# Show sample nodes
print("\nSample module names (first 10):")
for node in list(graph.nodes())[:10]:
    print(f"  - {node}")

# Step 4: Calculate metrics
print("\n[4/4] Calculating metrics...")
metrics = calculate_metrics(graph)
print(f"✅ Metrics calculated for {len(metrics)} modules")

# Show distribution
in_degrees = [m['in_degree'] for m in metrics.values()]
print(f"\nIn-degree statistics:")
print(f"  Max: {max(in_degrees) if in_degrees else 0}")
print(f"  Min: {min(in_degrees) if in_degrees else 0}")
print(f"  Avg: {sum(in_degrees)/len(in_degrees) if in_degrees else 0:.2f}")

# Rank brents
print("\nRanking Brents...")
top_brents = rank_brents(metrics, top_percentage=0.05)
print(f"✅ Found {len(top_brents)} top Brents")

if top_brents:
    print("\nTop 5 Brents:")
    for i, (module, score, data) in enumerate(top_brents[:5], 1):
        print(f"{i}. {module}: score={score:.2f}, in_degree={data['in_degree']}")

## Section 3: Debug Import Matching

Investigate why dependencies aren't being matched correctly.

In [ ]:
# Debug: Check what imports look like vs module names
from brent.graph_builder import _find_matching_modules

print("=" * 60)
print("IMPORT MATCHING DEBUG")
print("=" * 60)

# Get all internal modules
all_modules = set(metrics.keys())

# Sample a file with imports
test_file = files[0]  # First Flask file
test_imports = extract_imports(test_file, project_path)
test_module = os.path.splitext(os.path.relpath(test_file, project_path))[0].replace(os.sep, ".")

print(f"\nTest file: {os.path.relpath(test_file, project_path)}")
print(f"Module name: {test_module}")
print(f"Extracted imports: {test_imports[:5]}")  # First 5

if test_imports:
    print("\nMatching analysis (first 3 imports):")
    for imp in test_imports[:3]:
        matches = _find_matching_modules(imp, all_modules)
        print(f"\n  Import: '{imp}'")
        print(f"  Matches: {len(matches)} found")
        if matches:
            for match in matches[:3]:
                print(f"    - {match}")
        else:
            print(f"    (no matches - checking why...)")
            # Check which modules contain this import as substring
            containing_modules = [m for m in all_modules if imp in m or m in imp]
            if containing_modules:
                print(f"    Modules containing substring '{imp}': {containing_modules[:3]}")

# Summary statistics
print("\n" + "=" * 60)
print("IMPORT MATCHING SUMMARY")
print("=" * 60)

matched_count = 0
unmatched_count = 0
all_imports_seen = set()

for f in files:
    imports = extract_imports(f, project_path)
    for imp in imports:
        if imp not in all_imports_seen:
            all_imports_seen.add(imp)
            matches = _find_matching_modules(imp, all_modules)
            if matches:
                matched_count += 1
            else:
                unmatched_count += 1

print(f"Unique imports found: {len(all_imports_seen)}")
print(f"Matched to internal modules: {matched_count}")
print(f"Unmatched: {unmatched_count}")
print(f"Match rate: {100*matched_count/(matched_count+unmatched_count) if matched_count+unmatched_count > 0 else 0:.1f}%")

# Sample unmatched imports
print("\nSample unmatched imports:")
unmatched_samples = []
for f in files:
    imports = extract_imports(f, project_path)
    for imp in imports:
        matches = _find_matching_modules(imp, all_modules)
        if not matches and imp not in [u for u, _ in unmatched_samples]:
            unmatched_samples.append((imp, _find_matching_modules))
            if len(unmatched_samples) >= 5:
                break
    if len(unmatched_samples) >= 5:
        break

for imp, _ in unmatched_samples:
    print(f"  - '{imp}'")

## Section 4: Unit Tests for Matching Functions

Test the improved import matching algorithm on known cases.

In [ ]:
# Test import matching with known cases
print("=" * 60)
print("UNIT TESTS FOR IMPORT MATCHING")
print("=" * 60)

test_cases = [
    # (import_name, modules, expected_matches, test_name)
    ("app", {"app", "app.models", "app.views", "utils"}, {"app", "app.models", "app.views"}, "Submodule matching"),
    ("flask.app", {"src.flask.app", "src.flask.config"}, {"src.flask.app"}, "Flask-style suffix matching"),
    ("config", {"src.app.config", "src.app.models"}, {"src.app.config"}, "Deep package matching"),
    ("models", {"models", "models.user", "models.post"}, {"models", "models.user", "models.post"}, "Exact and submodule"),
]

passed = 0
failed = 0

for import_name, modules, expected, test_name in test_cases:
    matches = set(_find_matching_modules(import_name, modules))
    success = matches == expected
    
    status = "✅ PASS" if success else "❌ FAIL"
    print(f"\n{status}: {test_name}")
    print(f"  Import: '{import_name}'")
    print(f"  Modules: {modules}")
    print(f"  Expected: {expected}")
    print(f"  Got: {matches}")
    
    if success:
        passed += 1
    else:
        failed += 1

print(f"\n{'='*60}")
print(f"Results: {passed} passed, {failed} failed")
print(f"Success rate: {100*passed/(passed+failed):.0f}%")

## Section 5: Test with Simple Synthetic Project

Create a small test project with known dependencies to validate the whole pipeline.

In [ ]:
# Create a test project structure
import tempfile
import shutil

print("=" * 60)
print("SYNTHETIC PROJECT TEST")
print("=" * 60)

# Create temp directory
test_dir = tempfile.mkdtemp(prefix="brent_test_")
print(f"\nCreating test project in: {test_dir}")

try:
    # Create simple test files with known dependencies
    test_files = {
        "models.py": "# Core models",
        "utils.py": "# Utilities",
        "config.py": "# Configuration",
        "api.py": "from . import models\nfrom . import config\n# API uses models and config",
        "views.py": "from . import models\nfrom . import utils\n# Views use models and utils",
        "main.py": "from . import api\nfrom . import views\n# Main app",
    }
    
    # Write test files
    for filename, content in test_files.items():
        with open(os.path.join(test_dir, filename), 'w') as f:
            f.write(content)
    
    print(f"✅ Created {len(test_files)} test files")
    
    # Analyze the test project
    test_files_list = get_python_files(test_dir)
    print(f"Found {len(test_files_list)} Python files")
    
    # Build graph
    test_graph = build_dependency_graph(test_files_list, extract_imports, test_dir)
    print(f"\nTest Graph:")
    print(f"  Nodes: {test_graph.number_of_nodes()}")
    print(f"  Edges: {test_graph.number_of_edges()}")
    
    # Show dependencies
    if test_graph.number_of_nodes() > 0:
        print("\nModule dependencies found:")
        for node in test_graph.nodes():
            in_deg = test_graph.in_degree(node)
            out_deg = test_graph.out_degree(node)
            if in_deg > 0 or out_deg > 0:
                print(f"  {node}: in_degree={in_deg}, out_degree={out_deg}")
        
        # Show edges
        print("\nDependency edges:")
        for src, dst in test_graph.edges():
            print(f"  {src} → {dst}")
    else:
        print("⚠️  No modules detected")
    
    # Calculate metrics
    test_metrics = calculate_metrics(test_graph)
    test_brents = rank_brents(test_metrics, top_percentage=0.5)
    
    print(f"\nTop Brents (top 50%): {len(test_brents)}")
    for i, (module, score, data) in enumerate(test_brents, 1):
        print(f"  {i}. {module}: score={score:.2f}, in_degree={data['in_degree']}")

finally:
    # Cleanup
    shutil.rmtree(test_dir)
    print(f"\n✅ Cleaned up test directory")

## Section 6: Test Results Summary and Recommendations

Analysis of test outcomes and next steps for fixing any issues.

In [ ]:
# Summary Report
print("=" * 70)
print("BRENT DETECTOR - TESTING SUMMARY REPORT")
print("=" * 70)

print("\n📊 TEST RESULTS:")
print(f"  ✅ Library imports: All successful")
print(f"  ✅ Flask project analysis: {len(files)} files found, {graph.number_of_nodes()} modules")
print(f"  ⚠️  Dependencies found: {graph.number_of_edges()} edges")
print(f"  ✅ Metrics calculated: {len(metrics)} modules processed")
print(f"  ✅ Brent ranking: {len(top_brents)} Brents identified")

print("\n🔍 ISSUES IDENTIFIED:")
if graph.number_of_edges() == 0:
    print("  1. Zero dependencies found in Flask project")
    print("     - Likely cause: Import matching not working correctly")
    print("     - Module names: src.flask.xxx")
    print("     - Imports extracted: flask.xxx, config, etc.")
    print("     - Fix applied: Improved suffix matching algorithm")
else:
    print(f"  ✅ {graph.number_of_edges()} dependencies successfully extracted!")

print("\n✨ IMPROVEMENTS MADE:")
print("  1. Enhanced _find_matching_modules() with suffix matching")
print("  2. Improved parser to handle relative imports better")
print("  3. Added matching for imports without full path prefix")

print("\n📋 NEXT STEPS:")
print("  1. Run: python -m cli.main data/flask (to test with improvements)")
print("  2. Check if dependencies are now being extracted")
print("  3. Generate reports with --all option")
print("  4. Compare results with unit tests")

print("\n✅ NOTEBOOK TESTING COMPLETE")
print("=" * 70)